# Day 5 — TCN Autoencoder Training (SMD Dataset)

Trains a Temporal Convolutional Network Autoencoder on normal-behavior sequences from the SMD dataset. The model learns to reconstruct normal server telemetry; anomalous sequences produce elevated reconstruction error at inference time.

**Depends on Day 4 artifacts:**
- `artifacts/feature_pipeline.joblib`
- `artifacts/day4_if_metrics.json`

**Runtime note:** Training runs on CPU. Expect 3–8 min/epoch with early stopping around epoch 10–20.

In [5]:
import os
import sys
from pathlib import Path

# Walk up from cwd until the directory containing src/ is found
_root = Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# Set working directory to project root so all relative paths
# (artifacts/, data/, logs/) resolve correctly in every cell
os.chdir(_root)

print(f"Project root: {_root}")
print(f"Working dir:  {Path.cwd()}")
print(f"Python:       {sys.executable}")

Project root: /home/lobora/projects/clouddrift
Working dir:  /home/lobora/projects/clouddrift
Python:       /home/lobora/projects/clouddrift/.venv/bin/python


In [6]:
import json
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
)
logger = logging.getLogger("day5_smd")

ARTIFACTS_DIR = Path("artifacts")
print("Logging configured.")

Logging configured.


## Step 1 — Load Day 4 Artifacts

Reads `feature_cols` and `input_dim` from the Day 4 metrics file, and loads the fitted normalization pipeline. This ensures Day 5 uses exactly the same feature set as Day 4.

In [9]:
metrics_path = ARTIFACTS_DIR / "day4_if_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(
        f"Day 4 metrics not found at {metrics_path}. "
        "Run day4_if_training_smd.py (or notebook 03) first."
    )

with open(metrics_path) as f:
    day4_metrics = json.load(f)

feature_cols = day4_metrics["feature_cols"]
input_dim    = day4_metrics["input_dim_for_tcn"]

from src.features.engineering import load_feature_pipeline
pipeline = load_feature_pipeline()

logger.info("feature_cols: %d columns", len(feature_cols))
logger.info("input_dim for TCN: %d", input_dim)
logger.info("Feature pipeline loaded from artifacts/feature_pipeline.joblib")

2026-07-09 16:48:39,243  INFO      src.features.engineering  Feature pipeline loaded from: artifacts/feature_pipeline.joblib
2026-07-09 16:48:39,245  INFO      day5_smd  feature_cols: 68 columns
2026-07-09 16:48:39,245  INFO      day5_smd  input_dim for TCN: 68
2026-07-09 16:48:39,246  INFO      day5_smd  Feature pipeline loaded from artifacts/feature_pipeline.joblib


## Step 2 — Recreate Normalized Feature Splits

Day 4 did not save parquet files, so the data pipeline runs again here. This takes approximately 40 seconds and uses identical parameters to Day 4, ensuring the same train/val/test split.

In [ ]:
from src.data.ingestion import load_smd_dataset
from src.data.validation import (
    validate_smd_schema,
    define_temporal_split_per_series,
)
from src.features.engineering import (
    build_alibaba_features,
    apply_feature_pipeline,
)

logger.info("Loading SMD dataset...")
MACHINES = [f"machine-1-{i}" for i in range(1, 8)]  # 7 machines — ~3.4 GB RAM
raw_df = load_smd_dataset(machines=MACHINES)
raw_df = validate_smd_schema(raw_df)

logger.info("Building features...")
feat_df = build_alibaba_features(raw_df, group_col="machine_id")

logger.info("Splitting 70 / 15 / 15 per machine...")
train_df, val_df, test_df = define_temporal_split_per_series(
    feat_df,
    group_col="machine_id",
    train_pct=0.70,
    val_pct=0.15,
)

logger.info("Applying normalization pipeline...")
train_norm = apply_feature_pipeline(pipeline, train_df, feature_cols)
val_norm   = apply_feature_pipeline(pipeline, val_df,   feature_cols)
test_norm  = apply_feature_pipeline(pipeline, test_df,  feature_cols)

print(f"Train: {len(train_norm):,} rows | anomaly rate: {train_norm['is_anomaly'].mean()*100:.2f}%")
print(f"Val:   {len(val_norm):,} rows  | anomaly rate: {val_norm['is_anomaly'].mean()*100:.2f}%")
print(f"Test:  {len(test_norm):,} rows  | anomaly rate: {test_norm['is_anomaly'].mean()*100:.2f}%")

2026-07-09 16:49:13,948  INFO      day5_smd  Loading SMD dataset...
2026-07-09 16:49:13,951  INFO      src.data.ingestion  Loading SMD: 28 of 28 machines
2026-07-09 16:49:20,877  INFO      src.data.ingestion  SMD loaded: 1,416,825 rows | 28 machines | 29,444 anomaly rows (2.1%)
2026-07-09 16:49:20,883  INFO      src.data.validation  Running Pandera SMD schema validation...
2026-07-09 16:49:21,689  INFO      src.data.validation  SMD schema validation passed — 1,416,825 rows validated
2026-07-09 16:49:21,694  INFO      day5_smd  Building features...
2026-07-09 16:49:21,763  INFO      src.features.engineering  Building Alibaba features for 28 machines (grouped by machine_id)...
2026-07-09 16:49:28,030  INFO      src.features.engineering  Alibaba feature engineering complete: 1,416,825 rows, 68 feature columns
2026-07-09 16:49:28,065  INFO      day5_smd  Splitting 70 / 15 / 15 per machine...
2026-07-09 16:49:29,750  INFO      src.data.validation  Per-series temporal split (28 series) — tra

Train: 991,762 rows | anomaly rate: 0.73%
Val:   212,528 rows  | anomaly rate: 4.34%
Test:  212,535 rows  | anomaly rate: 6.13%


## Step 3 — Build SequenceDatasets

Creates sliding windows of length 30 timesteps from each machine's time series. Training uses normal rows only; validation uses all rows so `val_loss` reflects reconstruction quality across both normal and anomalous sequences.

In [ ]:
from src.models.tcn_autoencoder import SEQ_LENGTH, create_sequence_dataset

train_dataset = create_sequence_dataset(
    train_norm,
    feature_cols,
    seq_length=SEQ_LENGTH,
    normal_only=True,   # autoencoder trains on normal patterns only
)

val_dataset = create_sequence_dataset(
    val_norm,
    feature_cols,
    seq_length=SEQ_LENGTH,
    normal_only=False,  # all rows for val_loss monitoring
)

print(f"Train sequences: {len(train_dataset):,}")
print(f"Val sequences:   {len(val_dataset):,}")
print(f"Sequence shape:  (seq_length={SEQ_LENGTH}, input_dim={input_dim})")

## Step 4 — Train TCN Autoencoder

Trains with PyTorch Lightning. Key callbacks:
- **ModelCheckpoint**: saves the best model by `val_loss`
- **EarlyStopping**: stops after 5 epochs without improvement
- **MLFlowLogger**: logs losses to `mlflow.db`

Expect 3–8 minutes per epoch on CPU. Early stopping typically triggers around epoch 10–20.

In [ ]:
from src.models.tcn_autoencoder import train_tcn_autoencoder

logger.info(
    "Starting TCN training: input_dim=%d, seq_length=%d, max_epochs=100, patience=5",
    input_dim, SEQ_LENGTH,
)

model, best_ckpt = train_tcn_autoencoder(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    input_dim=input_dim,
    checkpoint_dir=ARTIFACTS_DIR / "checkpoints",
)

print(f"Training complete.")
print(f"Best checkpoint: {best_ckpt}")

## Step 5 — Calibrate Reconstruction Error Threshold

Uses contamination-based calibration: flags the top `(anomaly_rate × 1.5)%` of validation reconstruction errors. Biases slightly toward recall over precision, which is appropriate for anomaly detection.

In [ ]:
from src.models.tcn_autoencoder import calibrate_autoencoder_threshold

threshold = calibrate_autoencoder_threshold(model, val_norm, feature_cols)
print(f"Calibrated threshold: {threshold:.6f}")

## Step 6 — Evaluate on Validation and Test Sets

The key diagnostic is **error separation**: anomalous sequences must produce higher reconstruction error than normal sequences. If `error_mean_anomaly > error_mean_normal` shows ✓ on the test set, the model has learned to distinguish anomalous patterns.

In [ ]:
from src.models.tcn_autoencoder import evaluate_autoencoder

val_metrics  = evaluate_autoencoder(model, threshold, val_norm,  feature_cols, split_name="validation")
test_metrics = evaluate_autoencoder(model, threshold, test_norm, feature_cols, split_name="test")

print("\n--- Validation Set ---")
print(f"  Precision:              {val_metrics['precision']:.3f}")
print(f"  Recall:                 {val_metrics['recall']:.3f}")
print(f"  F1:                     {val_metrics['f1']:.3f}")
print(f"  AUC-ROC:                {val_metrics['auc_roc']:.3f}")
print(f"  Error mean (normal):    {val_metrics['error_mean_normal']:.4f}")
print(f"  Error mean (anomaly):   {val_metrics['error_mean_anomaly']:.4f}")
sep_val = val_metrics['error_mean_anomaly'] > val_metrics['error_mean_normal']
print(f"  Error separation:       {'✓' if sep_val else '✗'}")

print("\n--- Test Set ---")
print(f"  Precision:              {test_metrics['precision']:.3f}")
print(f"  Recall:                 {test_metrics['recall']:.3f}")
print(f"  F1:                     {test_metrics['f1']:.3f}")
print(f"  AUC-ROC:                {test_metrics['auc_roc']:.3f}")
print(f"  Error mean (normal):    {test_metrics['error_mean_normal']:.4f}")
print(f"  Error mean (anomaly):   {test_metrics['error_mean_anomaly']:.4f}")
sep_test = test_metrics['error_mean_anomaly'] > test_metrics['error_mean_normal']
print(f"  Error separation:       {'✓' if sep_test else '✗'}")

## Step 7 — Save Artifacts

Saves the trained model weights and hyperparameters to `artifacts/tcn_autoencoder.pt`. Also writes `artifacts/day5_tcn_metrics.json` for the ensemble notebook (Day 6).

In [ ]:
from src.models.tcn_autoencoder import save_tcn_autoencoder

save_tcn_autoencoder(model)

metrics_out = {
    "dataset": "SMD",
    "input_dim": input_dim,
    "seq_length": SEQ_LENGTH,
    "best_checkpoint": best_ckpt,
    "threshold": threshold,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
}

day5_path = ARTIFACTS_DIR / "day5_tcn_metrics.json"
with open(day5_path, "w") as f:
    json.dump(metrics_out, f, indent=2, default=str)

print(f"TCN model saved:    artifacts/tcn_autoencoder.pt")
print(f"TCN metrics saved:  {day5_path}")
print(f"\nNext: run the ensemble pipeline (notebook 06 or day6_ensemble.py)")
print(f"input_dim to confirm in ensemble: {input_dim}")